In [ ]:
!pip install datasets pandas

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd

# datasets>=4.0 no longer runs loading scripts (liar.py). Two script-free options:
try:
    # 1) Hugging Face's auto-converted Parquet copy of the dataset (no script needed)
    ds = load_dataset("ucsbnlp/liar", revision="refs/convert/parquet")
except Exception as e:
    print(f"Parquet branch unavailable ({type(e).__name__}); falling back to original TSV files")
    # 2) Original LIAR release (Wang, 2017)
    !wget -q -nc https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
    !unzip -q -o liar_dataset.zip -d liar_dataset
    cols = ["id", "label", "statement", "subject", "speaker", "job_title", "state_info",
            "party_affiliation", "barely_true_counts", "false_counts", "half_true_counts",
            "mostly_true_counts", "pants_on_fire_counts", "context"]
    read = lambda f: pd.read_csv(f"liar_dataset/{f}", sep="\t", header=None, names=cols, quoting=3)
    ds = DatasetDict({
        "train": Dataset.from_pandas(read("train.tsv")),
        "validation": Dataset.from_pandas(read("valid.tsv")),
        "test": Dataset.from_pandas(read("test.tsv")),
    })

print(ds)
# DatasetDict with splits: train (~10.2k), validation (~1.3k), test (~1.3k)

print(ds["train"][0])

Parquet branch unavailable (DatasetNotFoundError); falling back to original TSV files
DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1283
    })
})
{'id': '2635.json', 'label': 'false', 's

In [ ]:
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

# The Parquet copy stores labels as integers (ClassLabel); the TSV fallback already has strings.
label_feature = ds["train"].features["label"]
if hasattr(label_feature, "names"):
    label_names = label_feature.names
    for df in (train_df, val_df, test_df):
        df["label_name"] = df["label"].map(lambda i: label_names[i])
else:
    label_names = sorted(train_df["label"].unique())
    for df in (train_df, val_df, test_df):
        df["label_name"] = df["label"]

print(label_names)
print(train_df[["statement", "speaker", "party_affiliation", "label_name"]].head())